# EnergyDB — Getting Started

This notebook shows how to use EnergyDB to manage energy asset hierarchies, edges, and time series.

EnergyDB extends [TimeDB](https://github.com/rebase-energy/timedb) with a hierarchical node structure, letting you persist [EnergyDataModel](https://github.com/rebase-energy/EnergyDataModel) trees (portfolios, sites, farms, assets) at arbitrary depth, connect them via typed edges (lines, transformers, pipes), and link them to time series.

**Canonical flow:** build an EDM tree with `TimeSeries` dataframes attached to the nodes, then call `edb.EnergyDataClient(td).write(portfolio)` once — structure, schema, and data persist in a single call.

## 1. Connect and create schema

In [1]:
import energydb as edb
from timedb import TimeDataClient

td = TimeDataClient()
client = edb.EnergyDataClient(td)
client.delete()   # clean slate
client.create()   # create energydb schema + tables

## 2. Build the portfolio as a single EDM tree

Everything — structure, series schema, and data — is declared in one tree. `edb.TimeSeries` carries its DataFrame inline. `edb.TimeSeriesDescriptor` is still available for schema-only series (no data attached).

In [2]:
import polars as pl
import numpy as np
from datetime import datetime, timezone, timedelta
from shapely.geometry import Point

base = datetime(2025, 1, 1, tzinfo=timezone.utc)
index = [base + timedelta(hours=i) for i in range(24)]


def _simple(values):
    return pl.DataFrame({"valid_time": index, "value": values})


df_demand   = _simple(np.clip(np.random.normal(12,  3, 24), 0, 20).tolist())
df_t01_pwr  = _simple(np.clip(np.random.normal(2.5, 0.8, 24), 0, 3.5).tolist())
df_t01_wind = _simple(np.clip(np.random.normal(8,   2, 24), 0, 25).tolist())
df_t02_pwr  = _simple(np.clip(np.random.normal(2.8, 0.7, 24), 0, 3.5).tolist())
df_pv       = _simple(np.clip(np.random.normal(5,   2, 24), 0, 10).tolist())

portfolio = edb.Portfolio(name="My Portfolio", members=[
    edb.Site(name="Offshore-1", geometry=Point(3.0, 55.0), timeseries=[
        edb.TimeSeries(df_demand, name="demand", unit="MW", data_type=edb.DataType.ACTUAL),
    ], members=[
        edb.WindTurbine(name="T01", capacity=3.5, hub_height=80, timeseries=[
            edb.TimeSeries(df_t01_pwr,  name="power",      unit="MW",  data_type=edb.DataType.ACTUAL),
            edb.TimeSeries(df_t01_wind, name="wind_speed", unit="m/s", data_type=edb.DataType.ACTUAL),
            edb.TimeSeriesDescriptor(name="power", unit="MW", data_type=edb.DataType.FORECAST,
                                     timeseries_type=edb.TimeSeriesType.OVERLAPPING),
        ]),
        edb.WindTurbine(name="T02", capacity=3.5, hub_height=80, timeseries=[
            edb.TimeSeries(df_t02_pwr, name="power", unit="MW", data_type=edb.DataType.ACTUAL),
            edb.TimeSeriesDescriptor(name="power", unit="MW", data_type=edb.DataType.FORECAST,
                                     timeseries_type=edb.TimeSeriesType.OVERLAPPING),
        ]),
    ]),
    edb.Site(name="Rooftop-1", geometry=Point(4.5, 52.0), members=[
        edb.PVSystem(name="PV01", capacity=10, surface_tilt=25, surface_azimuth=180, timeseries=[
            edb.TimeSeries(df_pv, name="power", unit="MW", data_type=edb.DataType.ACTUAL),
        ]),
        edb.Battery(name="B01", storage_capacity=1000, max_charge=500),
    ]),
])

print(portfolio.to_tree())

Portfolio('My Portfolio')
├── Site('Offshore-1')
│   ├── WindTurbine('T01')
│   └── WindTurbine('T02')
└── Site('Rooftop-1')
    ├── PVSystem('PV01')
    │   └── PVArray()
    └── Battery('B01')


## 3. Persist it — one call

`client.write(portfolio)` upserts every node, registers every series declared on `timeseries=[…]`, and bulk-writes every attached `TimeSeries` dataframe in a single timedb round-trip. It's idempotent — safe to re-run.

In [3]:
root_id = client.write(portfolio)
print(f"Portfolio root node_id: {root_id}")

Portfolio root node_id: 1


## 4. Read with the fluent API

In [4]:
df = client.node("My Portfolio").node("Offshore-1").node("T01").read(
    data_type="actual", name="power",
    start_valid=datetime(2025, 1, 1, tzinfo=timezone.utc),
)
print("Single asset read:")
print(df)

Single asset read:
shape: (24, 10)
┌───────────┬──────┬────────────────┬──────────┬───┬─────────┬──────┬─────────────┬────────────────┐
│ series_id ┆ unit ┆ valid_time     ┆ value    ┆ … ┆ node_id ┆ node ┆ node_type   ┆ path           │
│ ---       ┆ ---  ┆ ---            ┆ ---      ┆   ┆ ---     ┆ ---  ┆ ---         ┆ ---            │
│ i64       ┆ str  ┆ datetime[μs,   ┆ f64      ┆   ┆ i64     ┆ str  ┆ str         ┆ str            │
│           ┆      ┆ UTC]           ┆          ┆   ┆         ┆      ┆             ┆                │
╞═══════════╪══════╪════════════════╪══════════╪═══╪═════════╪══════╪═════════════╪════════════════╡
│ 180002    ┆ MW   ┆ 2025-01-01     ┆ 2.65125  ┆ … ┆ 3       ┆ T01  ┆ WindTurbine ┆ My Portfolio/O │
│           ┆      ┆ 00:00:00 UTC   ┆          ┆   ┆         ┆      ┆             ┆ ffshore-1/T01  │
│ 180002    ┆ MW   ┆ 2025-01-01     ┆ 2.37471  ┆ … ┆ 3       ┆ T01  ┆ WindTurbine ┆ My Portfolio/O │
│           ┆      ┆ 01:00:00 UTC   ┆          ┆   ┆    

In [5]:
# Subtree read — all actuals for 'power' across the whole portfolio
df = client.node("My Portfolio").read(
    data_type="actual", name="power",
    start_valid=datetime(2025, 1, 1, tzinfo=timezone.utc),
)
print("Portfolio-wide read:")
print(df.head(10))

Portfolio-wide read:
shape: (10, 10)
┌───────────┬──────┬────────────────┬──────────┬───┬─────────┬──────┬─────────────┬────────────────┐
│ series_id ┆ unit ┆ valid_time     ┆ value    ┆ … ┆ node_id ┆ node ┆ node_type   ┆ path           │
│ ---       ┆ ---  ┆ ---            ┆ ---      ┆   ┆ ---     ┆ ---  ┆ ---         ┆ ---            │
│ i64       ┆ str  ┆ datetime[μs,   ┆ f64      ┆   ┆ i64     ┆ str  ┆ str         ┆ str            │
│           ┆      ┆ UTC]           ┆          ┆   ┆         ┆      ┆             ┆                │
╞═══════════╪══════╪════════════════╪══════════╪═══╪═════════╪══════╪═════════════╪════════════════╡
│ 180002    ┆ MW   ┆ 2025-01-01     ┆ 2.65125  ┆ … ┆ 3       ┆ T01  ┆ WindTurbine ┆ My Portfolio/O │
│           ┆      ┆ 00:00:00 UTC   ┆          ┆   ┆         ┆      ┆             ┆ ffshore-1/T01  │
│ 180002    ┆ MW   ┆ 2025-01-01     ┆ 2.37471  ┆ … ┆ 3       ┆ T01  ┆ WindTurbine ┆ My Portfolio/O │
│           ┆      ┆ 01:00:00 UTC   ┆          ┆   ┆  

## 5. Filter subtrees by type

In [6]:
df = client.node("My Portfolio").find(type="WindTurbine").read(
    data_type="actual", name="power",
    start_valid=datetime(2025, 1, 1, tzinfo=timezone.utc),
)
print("All WindTurbine actuals:")
print(df.head(10))

All WindTurbine actuals:
shape: (10, 10)
┌───────────┬──────┬────────────────┬──────────┬───┬─────────┬──────┬─────────────┬────────────────┐
│ series_id ┆ unit ┆ valid_time     ┆ value    ┆ … ┆ node_id ┆ node ┆ node_type   ┆ path           │
│ ---       ┆ ---  ┆ ---            ┆ ---      ┆   ┆ ---     ┆ ---  ┆ ---         ┆ ---            │
│ i64       ┆ str  ┆ datetime[μs,   ┆ f64      ┆   ┆ i64     ┆ str  ┆ str         ┆ str            │
│           ┆      ┆ UTC]           ┆          ┆   ┆         ┆      ┆             ┆                │
╞═══════════╪══════╪════════════════╪══════════╪═══╪═════════╪══════╪═════════════╪════════════════╡
│ 180002    ┆ MW   ┆ 2025-01-01     ┆ 2.65125  ┆ … ┆ 3       ┆ T01  ┆ WindTurbine ┆ My Portfolio/O │
│           ┆      ┆ 00:00:00 UTC   ┆          ┆   ┆         ┆      ┆             ┆ ffshore-1/T01  │
│ 180002    ┆ MW   ┆ 2025-01-01     ┆ 2.37471  ┆ … ┆ 3       ┆ T01  ┆ WindTurbine ┆ My Portfolio/O │
│           ┆      ┆ 01:00:00 UTC   ┆          ┆  

In [7]:
print("Children of My Portfolio:")
for child in client.node("My Portfolio").children():
    print(f"  {child['node_type']}: {child['name']}")

print("\nAll WindTurbines under the portfolio:")
for desc in client.node("My Portfolio").descendants(type="WindTurbine"):
    print(f"  {desc['name']} (node_id={desc['node_id']})")

Children of My Portfolio:
  Site: Offshore-1
  Site: Rooftop-1

All WindTurbines under the portfolio:
  T01 (node_id=3)
  T02 (node_id=4)


## 6. Reconstruct the EDM tree from the database

In [8]:
tree = client.get_tree("My Portfolio", include_series=True)
print(tree)

Portfolio('My Portfolio')
├── Site('Offshore-1')
│   ├── WindTurbine('T01')
│   └── WindTurbine('T02')
└── Site('Rooftop-1')
    ├── PVSystem('PV01')
    │   ├── PVArray()
    │   └── PVArray('PVArray')
    └── Battery('B01')


In [9]:
# Series attached to T01
t01 = tree.children()[0].children()[0]  # Offshore-1 → T01
print(f"{t01.name} series:")
for ts in t01.timeseries or []:
    print(f"  {ts.name} ({ts.data_type}, {ts.timeseries_type})")

T01 series:
  power (ACTUAL, FLAT)
  wind_speed (ACTUAL, FLAT)
  power (FORECAST, OVERLAPPING)


## 7. Also supported: imperative / fluent mutations

Beyond the declarative tree write, you can add, rename, or update individual nodes.

In [10]:
p = client.node("My Portfolio")

# Add a new turbine (with inline data) via declarative write under an existing parent
df_t03 = _simple(np.clip(np.random.normal(3.2, 0.6, 24), 0, 4.0).tolist())
client.write(
    edb.WindTurbine(name="T03", capacity=4.0, hub_height=90, timeseries=[
        edb.TimeSeries(df_t03, name="power", unit="MW", data_type=edb.DataType.ACTUAL),
    ]),
    parent="Offshore-1",
)

# Rename + update property
p.node("Offshore-1").node("T01").rename("T01-A")
p.node("Offshore-1").node("T01-A").update(properties={"capacity": 4.5})

print("Offshore-1 children after mutations:")
for child in p.node("Offshore-1").children():
    print(f"  {child['node_type']}: {child['name']} — {child['properties']}")

Offshore-1 children after mutations:
  WindTurbine: T01-A — {'capacity': 4.5, 'hub_height': 80}
  WindTurbine: T02 — {'capacity': 3.5, 'hub_height': 80}
  WindTurbine: T03 — {'capacity': 4.0, 'hub_height': 90}


## 8. Bulk manifest I/O

For high-performance reads/writes across many nodes, use manifest-based bulk I/O. Manifests can use `node_id` or `path` columns.

In [11]:
manifest = pl.DataFrame({
    "path": [
        "My Portfolio/Offshore-1/T01-A",
        "My Portfolio/Offshore-1/T02",
        "My Portfolio/Rooftop-1/PV01",
    ],
    "data_type": ["actual", "actual", "actual"],
    "name":    ["power",  "power",  "power"],
})

df = client.read(manifest, start_valid=datetime(2025, 1, 1, tzinfo=timezone.utc))
print(df.head(10))

shape: (10, 10)
┌───────────┬──────┬────────────────┬──────────┬───┬─────────┬───────┬─────────────┬───────────────┐
│ series_id ┆ unit ┆ valid_time     ┆ value    ┆ … ┆ node_id ┆ node  ┆ node_type   ┆ path          │
│ ---       ┆ ---  ┆ ---            ┆ ---      ┆   ┆ ---     ┆ ---   ┆ ---         ┆ ---           │
│ i64       ┆ str  ┆ datetime[μs,   ┆ f64      ┆   ┆ i64     ┆ str   ┆ str         ┆ str           │
│           ┆      ┆ UTC]           ┆          ┆   ┆         ┆       ┆             ┆               │
╞═══════════╪══════╪════════════════╪══════════╪═══╪═════════╪═══════╪═════════════╪═══════════════╡
│ 180002    ┆ MW   ┆ 2025-01-01     ┆ 2.65125  ┆ … ┆ 3       ┆ T01-A ┆ WindTurbine ┆ My Portfolio/ │
│           ┆      ┆ 00:00:00 UTC   ┆          ┆   ┆         ┆       ┆             ┆ Offshore-1/T0 │
│           ┆      ┆                ┆          ┆   ┆         ┆       ┆             ┆ 1-A           │
│ 180002    ┆ MW   ┆ 2025-01-01     ┆ 2.37471  ┆ … ┆ 3       ┆ T01-A ┆ Wind

## 9. Edges — typed links between nodes

Edges model grid topology: lines, transformers, pipes, interconnections, etc. They link two nodes and can carry their own time series (e.g. power flow).

In [12]:
# Create grid nodes
bus_a_id = client.create_node(edb.JunctionPoint(name="BusA"), parent="Offshore-1")
bus_b_id = client.create_node(edb.JunctionPoint(name="BusB"), parent="Offshore-1")

# Create an edge between them
line_id = client.create_edge(
    edb.Line(name="Cable-1", capacity=500),
    from_node=bus_a_id,
    to_node=bus_b_id,
)
print(f"Created Line edge_id={line_id}")

# Read it back
line = client.get_edge(id=line_id)
print(f"  type={type(line).__name__}, name={line.name}, capacity={line.capacity}")
print(f"  from_entity={line.from_entity.target}")
print(f"  to_entity={line.to_entity.target}")

# Query all edges within a subtree
edges = client.query_edges(type="Line", within="Offshore-1")
print(f"\nLines within Offshore-1: {len(edges)}")

# Register and write time series on an edge
scope = client.edge(id=line_id)
scope.register_series(name="power_flow", unit="MW", data_type="actual")

df_flow = _simple(np.clip(np.random.normal(200, 50, 24), 0, 500).tolist())
scope.write(df_flow, name="power_flow", data_type="actual")

# Read it back
df = scope.read(data_type="actual", name="power_flow")
print(f"\nPower flow series: {df.height} rows")
print(df.head(5))

# Navigate from edge to its endpoint nodes
print(f"\nFrom-node scope: {client.edge(id=line_id).from_node()}")
print(f"To-node scope: {client.edge(id=line_id).to_node()}")

Created Line edge_id=1
  type=Line, name=Cable-1, capacity=500
  from_entity=My Portfolio/Offshore-1/BusA
  to_entity=My Portfolio/Offshore-1/BusB

Lines within Offshore-1: 1

Power flow series: 24 rows
shape: (5, 11)
┌───────────┬──────┬─────────────┬────────────┬───┬─────────┬───────────┬─────────────┬────────────┐
│ series_id ┆ unit ┆ valid_time  ┆ value      ┆ … ┆ edge    ┆ edge_type ┆ from_node   ┆ to_node    │
│ ---       ┆ ---  ┆ ---         ┆ ---        ┆   ┆ ---     ┆ ---       ┆ ---         ┆ ---        │
│ i64       ┆ str  ┆ datetime[μs ┆ f64        ┆   ┆ str     ┆ str       ┆ str         ┆ str        │
│           ┆      ┆ , UTC]      ┆            ┆   ┆         ┆           ┆             ┆            │
╞═══════════╪══════╪═════════════╪════════════╪═══╪═════════╪═══════════╪═════════════╪════════════╡
│ 180009    ┆ MW   ┆ 2025-01-01  ┆ 276.087832 ┆ … ┆ Cable-1 ┆ Line      ┆ My Portfoli ┆ My Portfol │
│           ┆      ┆ 00:00:00    ┆            ┆   ┆         ┆           ┆ o

/home/kunruh/src/rebase-energy/energydb/energydb/scope.py:726: UserWarning: Inserting dimensionless values into series with unit 'MW' (series_id=180009). Values stored as-is without conversion.
  return self._td.write(write_df, series_col="series_id", **timedb_kwargs)


## 10. Cleanup

In [13]:
client.delete()